# 🎵 Hábitos de escucha de música: Springfield vs. Shelbyville

**Autor:** Sebastián Camilo Guarín Novoa · [LinkedIn](https://www.linkedin.com/in/sebastian-guarin/)

## Contexto

Una plataforma de streaming de música quiere saber si sus usuarios escuchan música de forma diferente según la ciudad en la que viven y el día de la semana. Para responderlo, uso registros reales de reproducciones de dos ciudades, **Springfield** y **Shelbyville**, de tres días: lunes, miércoles y viernes.

**Hipótesis:** la actividad de los usuarios cambia según el día de la semana, y ese patrón es distinto en cada ciudad.

**Etapas del análisis:**
1. Descripción de los datos
2. Preprocesamiento (encabezados, valores ausentes y duplicados)
3. Prueba de la hipótesis

## 1. Descripción de los datos

In [1]:
# Importar pandas
import pandas as pd

In [2]:
# Leer el archivo y almacenarlo en df
df = pd.read_csv("music_project_en.csv")

In [3]:
# Obtener las 10 primeras filas de la tabla df
df.head(10)

,userID,Track,artist,genre,City,time,Day
0,FFB692EC,Kamigata To Boots,The Mass Missile,rock,Shelbyville,20:28:33,Wednesday
1,55204538,Delayed Because of Accident,Andreas Rönnberg,rock,Springfield,14:07:09,Friday
2,20EC38,Funiculì funiculà,Mario Lanza,pop,Shelbyville,20:58:07,Wednesday
3,A3DD03C9,Dragons in the Sunset,Fire + Ice,folk,Shelbyville,08:37:09,Monday
4,E2DC1FAE,Soul People,Space Echo,dance,Springfield,08:34:34,Monday
5,842029A1,Chains,Obladaet,rusrap,Shelbyville,13:09:41,Friday
6,4CB90AA5,True,Roman Messer,dance,Springfield,13:00:07,Wednesday
7,F03E1C1F,Feeling This Way,Polina Griffith,dance,Springfield,20:47:49,Wednesday
8,8FA1D3BE,L’estate,Julia Dalia,ruspop,Springfield,09:17:40,Friday
9,E772D5C0,Pessimist,NaN,dance,Shelbyville,21:20:49,Wednesday


In [4]:
# Obtener la información general sobre nuestros datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65079 entries, 0 to 65078
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0     userID  65079 non-null  str  
 1   Track     63736 non-null  str  
 2   artist    57512 non-null  str  
 3   genre     63881 non-null  str  
 4     City    65079 non-null  str  
 5   time      65079 non-null  str  
 6   Day       65079 non-null  str  
dtypes: str(7)
memory usage: 3.5 MB


**Observaciones.** La tabla tiene siete columnas, todas de tipo `object`:

- `userID`: identificador del usuario
- `Track`: título de la canción
- `artist`: nombre del artista
- `genre`: género de la pista
- `City`: ciudad del usuario
- `time`: hora exacta de la reproducción
- `Day`: día de la semana

Los encabezados tienen tres problemas de estilo: mezclan mayúsculas y minúsculas, algunos tienen espacios al principio o al final, y `userID` no sigue la convención snake_case.

Además, las columnas `Track`, `artist` y `genre` tienen valores ausentes y hay filas duplicadas. Lo compruebo a continuación.

In [5]:
print(df. duplicated().sum())
df.sample(5)

3826


,userID,Track,artist,genre,City,time,Day
33085,B5868FBB,Rimnicu (chant de Noël Roumain),Christmas Hits,ruspop,Springfield,14:04:31,Friday
37653,E7901FEA,Behind the Lie,Surface 2 Air,rock,Springfield,14:11:15,Monday
6107,1CA721A1,"Violin Sonata No. 5 in F Major Op. 24 ""Spring""...",Jonathan Carney,classical,Springfield,21:18:53,Monday
6339,8AE6231E,Obnimi,Okean Elzi,rusrap,Shelbyville,09:09:07,Wednesday
8740,F7336427,Playinwitme,Kyle,dance,Springfield,14:55:13,Monday


## 2. Preprocesamiento de datos

### 2.1 Estilo de los encabezados

In [6]:
# Muestra los nombres de las columnas
df.columns

Index(['  userID', 'Track', 'artist', 'genre', '  City  ', 'time', 'Day'], dtype='str')

Paso todos los nombres a minúsculas, quito los espacios y renombro `userid` a `user_id`.

In [7]:
# Bucle en los encabezados poniendo todo en minúsculas
new_col_names = []
for column in df.columns:
    name_lowered = column.lower()
    new_col_names.append(name_lowered)

df.columns = new_col_names
df.columns

Index(['  userid', 'track', 'artist', 'genre', '  city  ', 'time', 'day'], dtype='str')

In [8]:
# Bucle en los encabezados eliminando los espacios
new_col_names = []
for column in df.columns:
    name_lowered = column.strip()
    new_col_names.append(name_lowered)

df.columns = new_col_names
df.columns

Index(['userid', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')

In [9]:
# Cambiar el nombre de la columna "userid"
columns_new = {"userid":"user_id"}
df.rename(columns = columns_new, inplace = True)


In [10]:
# Comprobar el resultado: la lista de encabezados
df.columns

Index(['user_id', 'track', 'artist', 'genre', 'city', 'time', 'day'], dtype='str')

### 2.2 Valores ausentes

In [11]:
# Calcular el número de valores ausentes
print(df.isna().sum())

user_id       0
track      1343
artist     7567
genre      1198
city          0
time          0
day           0
dtype: int64


Los valores ausentes en `track` y `artist` no afectan la hipótesis, que se basa en la cantidad de reproducciones. Los de `genre` sí podrían afectar una comparación de gustos musicales, pero no hay forma de recuperarlos. Por eso reemplazo los ausentes de las tres columnas con `'unknown'`, en lugar de eliminar esas filas y perder reproducciones válidas.

In [12]:
# Reemplazar los valores ausentes con 'unknown'
cols_to_replace = ['track', 'artist', 'genre']
for column in cols_to_replace:
    df[column] = df[column].fillna('unknown')

In [13]:
# Contar valores ausentes
print(df.isna().sum())

user_id    0
track      0
artist     0
genre      0
city       0
time       0
day        0
dtype: int64


### 2.3 Duplicados explícitos

In [14]:
# Contar duplicados explícitos
df.duplicated().sum()

np.int64(3826)

In [15]:
# Eliminar duplicados explícitos
df.drop_duplicates(inplace=True)

In [16]:
# Comprobar de nuevo si hay duplicados

df.duplicated().sum()

np.int64(0)

### 2.4 Duplicados implícitos en `genre`

Reviso la lista de géneros únicos, ordenada alfabéticamente, para encontrar nombres escritos de distintas formas.

In [17]:
# Inspeccionar los nombres de géneros únicos
print(sorted(df['genre'].unique()))
print('Géneros únicos:', df['genre'].nunique())

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

El género *hiphop* aparece también como `hip`, `hop` y `hip-hop`. Creo una función que reemplaza esas variantes por el nombre correcto.

In [18]:
# Función para reemplazar duplicados implícitos
def replace_wrong_genres(wrong_genres, correct_genre):
    for wg in wrong_genres:
        df['genre'] = df['genre'].replace(wg, correct_genre)

In [19]:
# Eliminar duplicados implícitos
replace_wrong_genres(["hip", "hop", "hip-hop"], "hiphop")

In [20]:
# Comprobación
print(sorted(df['genre'].unique()))
print('Géneros únicos:', df['genre'].nunique())
print('Duplicados explícitos:', df.duplicated().sum())

['acid', 'acoustic', 'action', 'adult', 'africa', 'afrikaans', 'alternative', 'ambient', 'americana', 'animated', 'anime', 'arabesk', 'arabic', 'arena', 'argentinetango', 'art', 'audiobook', 'avantgarde', 'axé', 'baile', 'balkan', 'beats', 'bigroom', 'black', 'bluegrass', 'blues', 'bollywood', 'bossa', 'brazilian', 'breakbeat', 'breaks', 'broadway', 'cantautori', 'cantopop', 'canzone', 'caribbean', 'caucasian', 'celtic', 'chamber', 'children', 'chill', 'chinese', 'choral', 'christian', 'christmas', 'classical', 'classicmetal', 'club', 'colombian', 'comedy', 'conjazz', 'contemporary', 'country', 'cuban', 'dance', 'dancehall', 'dancepop', 'dark', 'death', 'deep', 'deutschrock', 'deutschspr', 'dirty', 'disco', 'dnb', 'documentary', 'downbeat', 'downtempo', 'drum', 'dub', 'dubstep', 'eastern', 'easy', 'electronic', 'electropop', 'emo', 'entehno', 'epicmetal', 'estrada', 'ethnic', 'eurofolk', 'european', 'experimental', 'extrememetal', 'fado', 'film', 'fitness', 'flamenco', 'folk', 'folklor

Duplicados explícitos: 0


**Observaciones del preprocesamiento.** Al principio había 3.826 duplicados explícitos, que eliminé. Después corregí los duplicados implícitos del género hiphop (`hip`, `hop`, `hip-hop`) con una función reutilizable que reemplaza los nombres mal escritos por `hiphop`.

## 3. Prueba de la hipótesis

Comparo el número de canciones reproducidas por ciudad y por día (lunes, miércoles y viernes).

In [21]:
# Contar las canciones reproducidas en cada ciudad
df.groupby(by='city')["track"].count()

city
Shelbyville    18512
Springfield    42741
Name: track, dtype: int64

En Shelbyville se reprodujeron 18.512 canciones y en Springfield 42.741.

In [22]:
# Calcular las canciones reproducidas en cada uno de los tres días
df.groupby(by='day')["track"].count()

day
Friday       21840
Monday       21354
Wednesday    18059
Name: track, dtype: int64

El viernes se reprodujeron 21.840 canciones, el lunes 21.354 y el miércoles 18.059.

Creo una función que cuenta las reproducciones para una combinación de día y ciudad.

In [23]:
def number_tracks(dia, ciudad):
    # Filtrar por día y por ciudad
    filtered_df = df.query('day == @dia')
    filtered_df = filtered_df.query('city == @ciudad')
    # Contar las reproducciones
    return filtered_df['user_id'].count()

In [24]:
# El número de canciones reproducidas en Springfield el lunes
number_tracks("Monday", "Springfield")

np.int64(15740)

In [25]:
# El número de canciones reproducidas en Shelbyville el lunes
number_tracks("Monday", "Shelbyville")

np.int64(5614)

In [26]:
# El número de canciones reproducidas en Springfield el miércoles
number_tracks("Wednesday", "Springfield")

np.int64(11056)

In [27]:
# El número de canciones reproducidas en Shelbyville el miércoles
number_tracks("Wednesday", "Shelbyville")

np.int64(7003)

In [28]:
# El número de canciones reproducidas en Springfield el viernes
number_tracks("Friday", "Springfield")

np.int64(15945)

In [29]:
# El número de canciones reproducidas en Shelbyville el viernes
number_tracks("Friday", "Shelbyville")

np.int64(5895)

### 3.1 Comparación en porcentajes

Springfield tiene más del doble de reproducciones que Shelbyville, así que comparar los números absolutos solo muestra que es una ciudad más grande. Para ver si el **patrón semanal** es distinto, calculo qué porcentaje de las reproducciones de cada ciudad ocurre cada día.

In [30]:
# Tabla de reproducciones por ciudad y día
tabla = df.pivot_table(index='city', columns='day', values='user_id', aggfunc='count')
tabla = tabla[['Monday', 'Wednesday', 'Friday']]
tabla

day,Monday,Wednesday,Friday
city,,,
Shelbyville,5614,7003,5895
Springfield,15740,11056,15945


In [31]:
# Porcentaje de las reproducciones de cada ciudad que ocurre cada día
(tabla.div(tabla.sum(axis=1), axis=0) * 100).round(1)

day,Monday,Wednesday,Friday
city,,,
Shelbyville,30.3,37.8,31.8
Springfield,36.8,25.9,37.3


**Resultado.** Los patrones semanales son opuestos:

- En **Springfield** la actividad se concentra el **lunes (36,8 %)** y el **viernes (37,3 %)**, y cae el miércoles (25,9 %).
- En **Shelbyville** pasa lo contrario: el día con más actividad es el **miércoles (37,8 %)**, y el lunes (30,3 %) y el viernes (31,8 %) están por debajo.

Comparar en porcentajes confirma que la diferencia no se debe solo a que Springfield tenga más usuarios: el día en que más se escucha música es distinto en cada ciudad.

## Conclusiones

**Los datos respaldan la hipótesis:** la actividad de los usuarios cambia según el día de la semana, y ese patrón es distinto en cada ciudad.

- En números absolutos, Springfield reproduce más del doble de canciones que Shelbyville (42.741 frente a 18.512). El máximo absoluto fue el viernes en Springfield (15.945 reproducciones).
- En porcentajes, Springfield escucha más música al inicio y al final de la semana (lunes y viernes), mientras que Shelbyville tiene su pico a mitad de semana (miércoles).
- Para la plataforma, esto significa que acciones como lanzamientos o recomendaciones deberían programarse según la ciudad, no con un calendario único.

### Limitaciones

Este análisis compara proporciones de forma descriptiva, sin una prueba estadística formal. Además, los datos vienen de una sola plataforma, por lo que no representan necesariamente a toda la población de cada ciudad.